# EU Education Expenditure Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.


In [6]:
# Load libraries for data download, processing, and visualization
import requests
import json
import pandas as pd
import altair as alt
import os

alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

In [7]:
# Download World Bank education expenditure JSON files for EU member states
countries = {
    'AUT': 'Austria',
    'BEL': 'Belgium',
    'BGR': 'Bulgaria',
    'HRV': 'Croatia',
    'CYP': 'Cyprus',
    'CZE': 'Czech Republic',
    'DNK': 'Denmark',
    'EST': 'Estonia',
    'FIN': 'Finland',
    'FRA': 'France',
    'DEU': 'Germany',
    'GRC': 'Greece',
    'HUN': 'Hungary',
    'IRL': 'Ireland',
    'ITA': 'Italy',
    'LVA': 'Latvia',
    'LTU': 'Lithuania',
    'LUX': 'Luxembourg',
    'MLT': 'Malta',
    'NLD': 'Netherlands',
    'POL': 'Poland',
    'PRT': 'Portugal',
    'ROU': 'Romania',
    'SVK': 'Slovakia',
    'SVN': 'Slovenia',
    'ESP': 'Spain',
    'SWE': 'Sweden'
}

indicator = 'SE.XPD.TOTL.GD.ZS'
base_url = 'https://api.worldbank.org/v2/country'
date_range = '1995:2023'

os.makedirs('../data', exist_ok=True)

saved_files = []
failed_requests = []

for country_code, country_name in countries.items():
    url = f"{base_url}/{country_code}/indicator/{indicator}?date={date_range}&format=json&per_page=100"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        country_name_formatted = country_name.replace(' ', '_').lower()
        filename = f"../data/eu_education_{country_name_formatted}_data.json"
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        saved_files.append(filename)
    else:
        failed_requests.append({'country': country_name, 'status_code': response.status_code})

summary_downloads = pd.DataFrame(
    {
        'saved_files': len(saved_files),
        'failed_requests': len(failed_requests)
    },
    index=[0]
)
summary_downloads


,saved_files,failed_requests
0,27,0


In [8]:
# Verify presence of downloaded JSON files
missing_files = []
for country_code, country_name in countries.items():
    country_name_formatted = country_name.replace(' ', '_').lower()
    filename = f"../data/eu_education_{country_name_formatted}_data.json"
    if not os.path.exists(filename):
        missing_files.append(filename)

verification_summary = pd.DataFrame({
    'missing_files': [len(missing_files)],
    'total_expected': [len(countries)]
})
verification_summary


,missing_files,total_expected
0,0,27


In [9]:
# Build Altair line charts per country from saved JSON data
charts = {}
chart_counts = []


def process_wb_data(filename):
    with open(filename, 'r') as f:
        data = json.load(f)

    records = []
    if len(data) > 1 and data[1]:
        for item in data[1]:
            if item['value'] is not None:
                records.append({
                    'year': int(item['date']),
                    'education_gdp_pct': item['value']
                })
    return records


for country_code, country_name in sorted(countries.items()):
    country_name_formatted = country_name.replace(' ', '_').lower()
    json_file = f"../data/eu_education_{country_name_formatted}_data.json"
    processed_data = process_wb_data(json_file)

    chart = alt.Chart(alt.Data(values=processed_data)).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('year:Q', title='Year', axis=alt.Axis(grid=False, format='d')),
        y=alt.Y('education_gdp_pct:Q', title='% of GDP', scale=alt.Scale(domain=[0, 10]), axis=alt.Axis(grid=False)),
        color=alt.value('orange'),
        tooltip=[
            alt.Tooltip('year:Q', title='Year'),
            alt.Tooltip('education_gdp_pct:Q', format='.2f', title='% of GDP')
        ]
    ).properties(
        width=270,
        height=210,
        title={
            'text': f'{country_name} - Education Expenditures',
            'subtitle': 'Source: World Bank API | 1995-2023',
            'anchor': 'start'
        }
    ).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=11,
        titleFontSize=12
    ).configure_title(
        fontSize=13
    ).interactive()

    charts[country_code] = chart
    chart_counts.append({'country': country_name, 'data_points': len(processed_data)})

chart_counts_df = pd.DataFrame(chart_counts)
chart_counts_df.head()


,country,data_points
0,Austria,27
1,Belgium,18
2,Bulgaria,22
3,Cyprus,23
4,Czech Republic,27


In [10]:
# Save chart specifications to the graphs folder
os.makedirs('../graphs', exist_ok=True)

saved_paths = []
for country_code, chart in charts.items():
    country_name = countries[country_code].replace(' ', '_').lower()
    filename = f"../graphs/eu_education_{country_name}.json"
    chart.save(filename)
    saved_paths.append(filename)

saved_paths_df = pd.Series(saved_paths, name='saved_files').head()
saved_paths_df


0           ../graphs/eu_education_austria.json
1           ../graphs/eu_education_belgium.json
2          ../graphs/eu_education_bulgaria.json
3            ../graphs/eu_education_cyprus.json
4    ../graphs/eu_education_czech_republic.json
Name: saved_files, dtype: object